# Studio 2 — Latihan di Colab (runner)

Notebook ini tipis: seluruh logika pelatihan ada di repo (`train.py`, `make_split.py`,
`evaluate.py`). Notebook hanya menyiapkan GPU, menyiapkan dataset dari Drive, menjalankan
`train.py`, lalu menguji hasilnya dengan skrip penilai asli.

Tenggat: Preview 1 Rabu 14 Okt 23:59 · Preview 2 Rabu 21 Okt · Final Minggu 25 Okt 23:59.

> Sebelum menjalankan: menu **Runtime > Change runtime type > T4 GPU**.

## 1. Variabel pengaturan

Semua yang bisa berubah ada di satu sel ini. Ubah di sini, jangan di sel lain.

- `CANDIDATE` — berkas arsitektur yang dilatih (punya `get_model()`/`get_transform()`).
- `OUT_WEIGHTS` — tujuan bobot, langsung ke Drive supaya selamat kalau sesi putus.
- `DATASET_ZIP` — zip dataset di Drive.
- `DATA_DIR` — tujuan ekstrak (disk lokal Colab, bukan Drive).
- `EPOCHS`, `BATCH_SIZE`, `LR`.

In [ ]:
import os

# Repo kelompok yang dipakai di Colab.
REPO_URL = "https://github.com/SigitKurnia05/DL-RA_studio2-klasifikasi-sampah.git"
BRANCH = "main"

# Berkas arsitektur yang dilatih dan tujuan bobotnya (langsung ke Drive).
CANDIDATE = "submission/model.py"
OUT_WEIGHTS = "/content/drive/MyDrive/Studio 2/Ckpt/model_effnet_v2s.pth"

# Dataset: zip di Drive, diekstrak ke disk lokal Colab.
DATASET_ZIP = "/content/drive/MyDrive/Studio 2/Dataset/train.zip"
DATA_DIR = "/content/train"

# Hyperparameter.
EPOCHS = 12
BATCH_SIZE = 32
LR = 3e-4
WORKERS = 2

print("repo       :", REPO_URL)
print("arsitektur :", CANDIDATE)
print("bobot      :", OUT_WEIGHTS)
print("dataset    :", DATASET_ZIP, "->", DATA_DIR)
print(f"latih      : {EPOCHS} epoch, batch {BATCH_SIZE}, lr {LR}, {WORKERS} worker")

## 2. Siapkan lingkungan

Mount Drive, clone repo (kalau belum ada), pasang paket tambahan, cek GPU dan versi torch.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

if not os.path.isdir("/content/repo/.git"):
    !git clone -b $BRANCH $REPO_URL /content/repo
%cd /content/repo
!git pull --ff-only

!pip install -q -r submission/requirements.txt

!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "TANPA GPU - set Runtime > Change runtime type > T4 GPU"
!python -c "import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())"

## 3. Ekstrak dataset ke disk lokal

Zip diekstrak ke `DATA_DIR` (disk lokal Colab). Membaca ribuan gambar kecil langsung dari Drive
jauh lebih lambat saat pelatihan, jadi ekstrak dulu.

Jumlah gambar per kelas dibandingkan dengan snapshot yang membekukan `splits/val.txt`
(2753 gambar: 523/521/365/673/671). Kalau berbeda, jangan lanjut melatih: datasetnya bukan
snapshot yang sama sehingga angkanya tidak sebanding.

In [ ]:
import pathlib
import zipfile

CLASSES = ["botol_plastik", "gelas_plastik", "kertas_kardus", "saset_kemasan", "styrofoam"]
EXPECT = {"botol_plastik": 523, "gelas_plastik": 521, "kertas_kardus": 365,
          "saset_kemasan": 673, "styrofoam": 671}


def cari_train(dasar):
    """Cari folder yang berisi lima subfolder kelas.

    Args:
        dasar: folder yang ditelusuri.

    Returns:
        list: daftar folder train yang cocok.
    """
    return [d for d in pathlib.Path(dasar).rglob("train")
            if d.is_dir() and all((d / c).is_dir() for c in CLASSES)]


zip_path = pathlib.Path(DATASET_ZIP)
root = pathlib.Path(DATA_DIR)
if not zip_path.is_file():
    raise SystemExit(f"[GAGAL] zip dataset tidak ditemukan: {zip_path}")

if not root.exists():
    root.mkdir(parents=True)
    print(f"ekstrak {zip_path.name} -> {root}")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(root)
else:
    print(f"{root} sudah ada, tidak diekstrak ulang")

ditemukan = cari_train(root)
if not ditemukan:
    raise SystemExit(f"[GAGAL] tidak ada folder train berisi lima kelas di {root}")

DATA = max(ditemukan, key=lambda d: sum(len(list((d / c).glob("*.jpg"))) for c in CLASSES))
jumlah = {c: len(list((DATA / c).glob("*.jpg"))) for c in CLASSES}
print("dataset:", DATA, "| total", sum(jumlah.values()), "gambar")
for c in CLASSES:
    tanda = "" if jumlah[c] == EXPECT[c] else f"  <-- beda dari snapshot ({EXPECT[c]})"
    print(f"  {c:16s} {jumlah[c]:5d}{tanda}")
if any(jumlah[c] != EXPECT[c] for c in CLASSES):
    print("\n[PERINGATAN] snapshot dataset berbeda; make_split.py --check akan menolak.")

## 4. Latih

Split validasi dibaca dari `splits/val.txt` yang sudah dibekukan di repo (tidak pernah dibuat
ulang di sini), jadi angka Colab dan angka laptop dihitung atas 413 gambar yang sama.

`train.py` mencetak per epoch: `train_loss`, `val_loss`, `val_macro_f1`, `val_acc`, `lr` saat itu,
dan waktu epoch; tanda **bintang** muncul saat rekor Macro-F1 baru. Riwayatnya ditulis ke
`runs/history.json` dan dibaca oleh sel visualisasi di bawah.

In [ ]:
print(f"latih {EPOCHS} epoch dari {CANDIDATE}")
!python train.py --model {CANDIDATE} --data {DATA} --out {OUT_WEIGHTS} --epochs {EPOCHS} --bs {BATCH_SIZE} --lr {LR} --workers {WORKERS}

## 5. Uji dengan skrip penilai asli

`evaluate.py` adalah skrip yang sama persis dengan yang dipakai asisten. Kandidat diuji di folder
terpisah (`/tmp/candidate`) supaya `submission/` tidak tersentuh sebelum hasilnya terbukti bagus.
Perhatikan juga waktu inferensinya: batas asisten 15 menit untuk seluruh validation set.

In [ ]:
import pathlib, shutil

cand = pathlib.Path("/tmp/candidate")
cand.mkdir(exist_ok=True)
shutil.copy(CANDIDATE, cand / "model.py")
shutil.copy(OUT_WEIGHTS, cand / "model.pth")
print("folder kandidat:", sorted(q.name for q in cand.iterdir()))

!python evaluate.py --submission /tmp/candidate --data ./my_val --out /content/repo/runs/eval_colab.json

## 6. Visualisasi hasil latihan

Sel ini membaca `runs/history.json` (ditulis `train.py`) lalu menggambar tiga hal:
kurva Train vs Val Loss, kurva Macro-F1, dan heatmap confusion matrix dari hasil evaluasi.
Ketiganya disimpan sebagai satu berkas `training_curves.png` di folder Ckpt Drive.

In [ ]:
import json, pathlib

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from matplotlib.ticker import MaxNLocator

hist_path = pathlib.Path("runs/history.json")
if not hist_path.is_file():
    raise SystemExit("[GAGAL] runs/history.json tidak ada - jalankan sel pelatihan lebih dulu")

hist = json.loads(hist_path.read_text(encoding="utf-8"))
ep = [h["epoch"] for h in hist]
tr = [h["train_loss"] for h in hist]
va = [h["val_loss"] for h in hist]
f1 = [h["val_macro_f1"] for h in hist]
acc = [h["val_acc"] for h in hist]

puncak = max(range(len(f1)), key=lambda i: f1[i])
print(f"epoch tercatat : {len(ep)}")
print(f"macro-F1 terbaik: {f1[puncak]:.4f} pada epoch {ep[puncak]}")
print(f"akurasi epoch itu: {acc[puncak]:.4f}")
print(f"waktu total    : {sum(h['detik'] for h in hist):.0f} detik "
      f"(rata-rata {np.mean([h['detik'] for h in hist]):.1f} detik/epoch)")

fig, ax = plt.subplots(1, 3, figsize=(19, 5.2))

ax[0].plot(ep, tr, "o-", color="#1f77b4", label="Train loss")
ax[0].plot(ep, va, "s-", color="#d62728", label="Val loss")
ax[0].set_title("Train vs Val Loss")
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("Loss (label smoothing 0,1)")
ax[0].legend()
ax[0].grid(alpha=0.3)
ax[0].xaxis.set_major_locator(MaxNLocator(integer=True))

ax[1].plot(ep, f1, "o-", color="#2ca02c", label="Val Macro-F1")
ax[1].plot(ep, acc, "^--", color="#7f7f7f", label="Val akurasi")
ax[1].axhline(0.92, color="#d62728", ls=":", lw=1.5, label="Baseline asisten ~0,92")
ax[1].scatter([ep[puncak]], [f1[puncak]], s=180, marker="*", color="#ff7f0e", zorder=5,
              label=f"Terbaik {f1[puncak]:.4f} (ep {ep[puncak]})")
ax[1].set_title("Macro-F1 validasi")
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel("Macro-F1 / akurasi")
ax[1].legend(fontsize=9)
ax[1].grid(alpha=0.3)
ax[1].xaxis.set_major_locator(MaxNLocator(integer=True))

laporan = pathlib.Path("runs/eval_colab.json")
if laporan.is_file():
    m = json.loads(laporan.read_text(encoding="utf-8"))
    cm = np.array(m["confusion_matrix"])
    # Urutan kelas dibaca dari kunci per_class_f1 pada laporan evaluate.py, jadi label
    # sumbu selalu mengikuti urutan kelas milik penilai tanpa perlu impor apa pun.
    label = list(m.get("per_class_f1", {})) or [f"kelas {i}" for i in range(len(cm))]
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=label, yticklabels=label, ax=ax[2])
    ax[2].set_title(f"Confusion matrix (macro-F1 {m['macro_f1']:.4f}, "
                    f"akurasi {m['accuracy']:.4f})")
else:
    ax[2].text(0.5, 0.5, "runs/eval_colab.json belum ada", ha="center", va="center")
    ax[2].set_title("Confusion matrix (belum tersedia)")
ax[2].set_xlabel("Prediksi")
ax[2].set_ylabel("Label asli")
plt.setp(ax[2].get_xticklabels(), rotation=30, ha="right")

fig.suptitle("Studio 2 — Klasifikasi Sampah: ringkasan latihan", fontsize=13)
fig.tight_layout()

keluar = pathlib.Path("/content/drive/MyDrive/Studio 2/Ckpt/training_curves.png")
keluar.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(keluar, dpi=150, bbox_inches="tight")
print("gambar disimpan:", keluar)
plt.show()

## 7. Ringkasan dan unduh bobot

Ukuran bobot diperiksa terhadap batas GitHub 100 MB per berkas (di atas itu harus lewat GitHub
Release atau LFS, dan tautannya ditulis di README).

In [ ]:
import hashlib, json, pathlib

bobot = pathlib.Path(OUT_WEIGHTS)
mb = bobot.stat().st_size / 2**20
print(f"{bobot}: {mb:.1f} MB (batas GitHub 100 MB, batas teknis 500 MB)")
print("sha256:", hashlib.sha256(bobot.read_bytes()).hexdigest())
if mb > 100:
    print("[CATATAN] di atas 100 MB: unggah lewat GitHub Release atau LFS, tautannya tulis di README")

laporan = pathlib.Path("runs/eval_colab.json")
if laporan.is_file():
    m = json.loads(laporan.read_text(encoding="utf-8"))
    print(f"macro-F1 {m['macro_f1']:.4f} | akurasi {m['accuracy']:.4f} | "
          f"{m['n_images']} gambar | {m['seconds']} detik")

from google.colab import files

files.download(OUT_WEIGHTS)
files.download("/content/drive/MyDrive/Studio 2/Ckpt/training_curves.png")

## 8. Setelah notebook selesai

1. Bobot sudah tersimpan di Drive (`Ckpt/`), jadi tidak perlu diunduh ulang kalau sesi Colab mati.
2. Taruh bobot itu di `submission/model.pth` di laptop, lalu uji sendiri:
   `python evaluate.py --submission ./submission --data ./my_val`.
3. Promosi ke repo = satu commit untuk dua berkas (`submission/model.py` dan
   `submission/model.pth`). Kalau salah satu tertinggal, `evaluate.py` gagal memuat
   (`strict=True`) dan submission tidak mendapat skor pada putaran itu.
4. Catat hasilnya di tabel README, lalu perbarui saat skor dari asisten diumumkan.